In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import pandas as pd
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from atow_estimation.paths import PROCESSED_DATA_DIR
from atow_estimation.paths import RAW_DATA_DIR

read data

In [3]:
full_data = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'processed_dataset.csv'))
aircraft_seats = pd.read_csv(os.path.join(RAW_DATA_DIR, 'aircraft_seats.csv'), sep=';')
iata = pd.read_csv(os.path.join(RAW_DATA_DIR, 'iata.csv'), sep=';')

add iata regions columns

In [4]:
unique_combinations_in_full_data = full_data[['adep', 'ades']].drop_duplicates()
iata_filtered = pd.merge(iata, unique_combinations_in_full_data, on=['adep', 'ades'], how='inner')
full_data = pd.merge(full_data, iata_filtered, on=['adep', 'ades'], how='inner')

merge with aircraft_seats to get seats and payload

In [5]:
full_data = pd.merge(full_data, aircraft_seats, on=['aircraftRegistration', 'IATAregionOrigen', 'IATAregionDestino'], how='left')

In [6]:
full_data

,flightKey,CFMUflightKey,callsign,adep,ades,ADEPLat,ADEPLong,ADESLat,ADESLong,routeType,...,mfc,oew,mtow_rates,IATAregionOrigen,IATAregionDestino,LoadFactor,seats,dateFrom,dateTo,Payload
0,17506376,15460777,BEL3732,LEMD,EBBR,40.47222,-3.560833,50.90139,4.484445,INTERNATIONAL,...,24210,42600,76880.0,Europe,Europe,85.3,180.0,2012-04-11,2024-01-08,15354.0
1,17506399,15461181,AAL94,KJFK,LEMD,40.64028,-73.778340,40.47222,-3.560833,INTERNATIONAL,...,171000,138000,266810.0,North America,Europe,81.0,273.0,2000-02-29,2016-05-01,22113.0
2,17506561,15460987,JAF3PV,EBAW,LEMI,51.18945,4.460278,37.80500,-1.118056,INTERNATIONAL,...,16153,27753,48990.0,Europe,Europe,85.3,112.0,2015-04-24,2016-10-19,9553.6
3,17506561,15460987,JAF3PV,EBAW,LEMI,51.18945,4.460278,37.80500,-1.118056,INTERNATIONAL,...,16153,27753,48990.0,Europe,Europe,85.3,112.0,2016-10-19,2023-04-11,9553.6
4,17506725,15461498,THY1857,LTFM,LEMD,41.27528,28.751940,40.47222,-3.560833,INTERNATIONAL,...,139000,122780,217000.0,Europe,Europe,85.3,289.0,2012-04-26,9999-12-31,24651.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81024,20475668,54315956,SWR271X,LPPT,LSGG,38.77417,-9.134167,46.23833,6.109445,OVERFLIGHT,...,21805,36809,58000.0,Europe,Europe,85.3,125.0,2016-12-21,2018-04-19,10662.5
81025,20475777,54315729,SWR191N,LEBL,LSZH,41.29694,2.078333,47.45806,8.548056,INTERNATIONAL,...,24210,42600,76880.0,Europe,Europe,85.3,174.0,2022-05-16,9999-12-31,14842.2
81026,20475777,54315729,SWR191N,LEBL,LSZH,41.29694,2.078333,47.45806,8.548056,INTERNATIONAL,...,24210,42600,76880.0,Europe,Europe,85.3,180.0,2013-03-20,2022-05-09,15354.0
81027,20475810,54314068,EIN49D,LPFR,EIDW,37.01445,-7.965833,53.42139,-6.270000,OVERFLIGHT,...,24210,42600,76880.0,Europe,Europe,85.3,174.0,2009-04-03,2023-01-27,14842.2


In [7]:
full_data[['seats', 'LoadFactor','Payload']].isnull().sum()

seats         13
LoadFactor    13
Payload       13
dtype: int64

In [9]:
full_data['date'] = full_data['ATOT_track'].str[:10]

In [10]:
full_data['dateFrom'] = full_data['dateFrom'].fillna(full_data['date'])

In [12]:
df_past_dates = full_data[full_data['dateFrom'] <= full_data['date']]
idx = df_past_dates.groupby('flightKey')['date'].idxmax()

full_data = df_past_dates.loc[idx]

In [13]:
full_data

,flightKey,CFMUflightKey,callsign,adep,ades,ADEPLat,ADEPLong,ADESLat,ADESLong,routeType,...,oew,mtow_rates,IATAregionOrigen,IATAregionDestino,LoadFactor,seats,dateFrom,dateTo,Payload,date
0,17506376,15460777,BEL3732,LEMD,EBBR,40.47222,-3.560833,50.90139,4.484445,INTERNATIONAL,...,42600,76880.0,Europe,Europe,85.3,180.0,2012-04-11,2024-01-08,15354.0,2022-01-01
1,17506399,15461181,AAL94,KJFK,LEMD,40.64028,-73.778340,40.47222,-3.560833,INTERNATIONAL,...,138000,266810.0,North America,Europe,81.0,273.0,2000-02-29,2016-05-01,22113.0,2022-01-01
2,17506561,15460987,JAF3PV,EBAW,LEMI,51.18945,4.460278,37.80500,-1.118056,INTERNATIONAL,...,27753,48990.0,Europe,Europe,85.3,112.0,2015-04-24,2016-10-19,9553.6,2022-01-01
4,17506725,15461498,THY1857,LTFM,LEMD,41.27528,28.751940,40.47222,-3.560833,INTERNATIONAL,...,122780,217000.0,Europe,Europe,85.3,289.0,2012-04-26,9999-12-31,24651.7,2022-01-01
5,17506733,15461851,SAS6201,EKCH,LEMG,55.61806,12.656110,36.67500,-4.499166,INTERNATIONAL,...,44300,73500.0,Europe,Europe,85.3,180.0,2017-05-02,2018-05-01,15354.0,2022-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81022,20475377,54315863,THY7BP,LPPT,LTFM,38.77417,-9.134167,41.27528,28.751940,OVERFLIGHT,...,50000,89400.0,Europe,Europe,85.3,182.0,2019-10-30,9999-12-31,15524.6,2022-12-31
81023,20475668,54315956,SWR271X,LPPT,LSGG,38.77417,-9.134167,46.23833,6.109445,OVERFLIGHT,...,36809,58000.0,Europe,Europe,85.3,125.0,2016-12-21,2016-12-30,10662.5,2022-12-31
81025,20475777,54315729,SWR191N,LEBL,LSZH,41.29694,2.078333,47.45806,8.548056,INTERNATIONAL,...,42600,76880.0,Europe,Europe,85.3,174.0,2022-05-16,9999-12-31,14842.2,2022-12-31
81027,20475810,54314068,EIN49D,LPFR,EIDW,37.01445,-7.965833,53.42139,-6.270000,OVERFLIGHT,...,42600,76880.0,Europe,Europe,85.3,174.0,2009-04-03,2023-01-27,14842.2,2022-12-31


In [14]:
full_data[['seats', 'Payload']].isnull().sum()

seats      13
Payload    13
dtype: int64

In [15]:
full_data[
    (full_data['seats'].isna()) &
    (full_data['Payload'].isna())][['aircraftType']]

,aircraftType
1470,A320
4183,A20N
6276,B77W
21458,B738
22997,A20N
31861,A320
44778,B738
50482,B38M
56879,B789
61349,BCS3


In [16]:
load_factor_rules = {
    'Africa': {'inter': 72.9, 'intra': 73.3},
    'Asia Pacific': {'inter': 84.9, 'intra': 84.9},
    'Europe': {'inter': 85.0, 'intra': 85.3},
    'South America': {'inter': 84.4, 'intra': 84.5},
    'Middle East': {'inter': 81.0, 'intra': 81.2},
    'North America': {'inter': 81.0, 'intra': 81.0}
}

In [17]:
nan_loadfactor_mask = full_data['LoadFactor'].isna()

for region_name, factors in load_factor_rules.items():
    # Rule 1: IATAregionOrigen = 'region_name' AND IATAregionDestino != 'region_name'
    inter_regional_condition = (full_data['IATAregionOrigen'] == region_name) & \
                               (full_data['IATAregionDestino'] != region_name) & \
                               nan_loadfactor_mask # Only apply to NaNs

    full_data.loc[inter_regional_condition, 'LoadFactor'] = factors['inter']

    # Rule 2: IATAregionOrigen = 'region_name' AND IATAregionDestino = 'region_name'
    intra_regional_condition = (full_data['IATAregionOrigen'] == region_name) & \
                               (full_data['IATAregionDestino'] == region_name) & \
                               nan_loadfactor_mask # Only apply to NaNs

    full_data.loc[intra_regional_condition, 'LoadFactor'] = factors['intra']


In [18]:
act_to_fill = full_data[
    (full_data['seats'].isna()) &
    (full_data['Payload'].isna())].aircraftType.unique()

In [19]:
for ac in act_to_fill:
    median_seats_for_ac = full_data[full_data['aircraftType'] == ac]['seats'].median()
    full_data.loc[
    (full_data['aircraftType'] == ac) & (full_data['seats'].isna()),
    'seats'
] = full_data.loc[
    (full_data['aircraftType'] == ac) & (full_data['seats'].isna()),
    'seats'
].fillna(median_seats_for_ac)

In [20]:
for ac in act_to_fill:
    mask_payload_nan = (full_data['aircraftType'] == ac) & (full_data['Payload'].isna())
    seats_for_calculation = full_data.loc[mask_payload_nan, 'seats']
    loadfactor_for_calculation = full_data.loc[mask_payload_nan, 'LoadFactor']
    calculated_payload_values = seats_for_calculation * loadfactor_for_calculation
    full_data.loc[mask_payload_nan, 'Payload'] = full_data.loc[mask_payload_nan, 'Payload'].fillna(calculated_payload_values)     

In [21]:
full_data.columns

Index(['flightKey', 'CFMUflightKey', 'callsign', 'adep', 'ades', 'ADEPLat',
       'ADEPLong', 'ADESLat', 'ADESLong', 'routeType',
       ...
       'oew', 'mtow_rates', 'IATAregionOrigen', 'IATAregionDestino',
       'LoadFactor', 'seats', 'dateFrom', 'dateTo', 'Payload', 'date'],
      dtype='object', length=107)

In [22]:
full_data = full_data.drop(columns=['IATAregionOrigen', 'IATAregionDestino', 'LoadFactor', 'dateFrom', 'dateTo', 'date'])

In [25]:
full_data.columns

Index(['flightKey', 'CFMUflightKey', 'callsign', 'adep', 'ades', 'ADEPLat',
       'ADEPLong', 'ADESLat', 'ADESLong', 'routeType',
       ...
       'CLIMB_vel_z_mean_60-90fl', 'CLIMB_vel_mod_variance_60-90fl',
       'CLIMB_vel_z_variance_60-90fl', 'mtow_openap', 'mlw', 'mfc', 'oew',
       'mtow_rates', 'seats', 'Payload'],
      dtype='object', length=101)

In [26]:
full_data.to_csv(os.path.join(PROCESSED_DATA_DIR, "processed_dataset.csv"), index=False)

In [27]:
full_data[['seats', 'Payload']].isnull().sum()

seats      0
Payload    0
dtype: int64

---

In [ ]:
def fill_mtow_rates(df, aircraftType):
    relevant_df = df[df["aircraftType"] == aircraftType].copy()
    if relevant_df["mtow_rates"].dropna().empty:
        print(
            f"No existing 'mtow_rates' for aircraft type: {aircraftType}. Cannot fill missing values."
        )
        return df

    most_common_mtow = relevant_df["mtow_rates"].mode().iloc[0]
    lookup_cols = ["aircraftType", "airlineCode", "adep", "ades"]
    similar_flights_lookup = relevant_df[
        (relevant_df["mtow_rates"] == most_common_mtow) & (relevant_df["mtow_rates"].notna())
    ][lookup_cols + ["mtow_rates"]].drop_duplicates()
    df = pd.merge(
        df, similar_flights_lookup, how="left", on=lookup_cols, suffixes=("_original", "_lookup")
    )

    df["mtow_rates"] = df["mtow_rates_original"].fillna(df["mtow_rates_lookup"])
    df = df.drop(columns=["mtow_rates_original", "mtow_rates_lookup"])
    return df

mtows_rates_weight = mtows_rates_weight.drop_duplicates()
full_data = full_data.merge(mtows_rates_weight, on='flightKey', how='left')

print('num of missing mtows:', len(full_data[full_data['mtow_rates'].isna()]))
# full_data[full_data['mtow_rates'].isna()][['flightKey', 'aircraftType', 'mtow_openap', 'mtow_rates']]
act = full_data[full_data['mtow_rates'].isna()].aircraftType.unique()

for ac in list(act):
    full_data = fill_mtow_rates(full_data, ac)
    
print('num of missing mtows:', len(full_data[full_data['mtow_rates'].isna()]))
# full_data[full_data['mtow_rates'].isna()][['flightKey', 'aircraftType', 'mtow_openap', 'mtow_rates']]    
full_data["mtow_rates"] = full_data["mtow_rates"].fillna(full_data["mtow_openap"])
full_data.to_csv(os.path.join(PROCESSED_DATA_DIR, "processed_dataset.csv"), index=False)

In [ ]:
act = full_data[full_data['mtow_rates'].isna()].aircraftType.unique()
# act
# check if set of aircraft types with missing mtow_rates are in set of aircraft types with non missing mtow_rates
# set(full_data[(full_data['mtow_rates'].notna()) & (full_data['aircraftType'].isin(act))].aircraftType.unique()) == set(act)
# non null mtow_rates for aircraft types that have null mtow_rates
data_with_nonull_mtow_rates = full_data[(full_data['mtow_rates'].notna()) & (full_data['aircraftType'].isin(act))][['flightKey', 'aircraftType', 'mtow_openap', 'mtow_rates']]
# data_with_nonull_mtow_rates
nunique_mtow_rates = data_with_nonull_mtow_rates.groupby('aircraftType')['mtow_rates'].nunique()
# nunique_mtow_rates[nunique_mtow_rates == 1].index
inconsistent_aircraft_types = nunique_mtow_rates[nunique_mtow_rates > 1]
# inconsistent_aircraft_types

array(['A320', 'B738', 'B772', 'B788', 'B789', 'E190', 'A321', 'B38M',
       'A20N', 'A319', 'BCS3', 'A333', 'BCS1', 'B737', 'E195'],
      dtype=object)